# ProCam Calibration using ArUco + Projected Checkerboard

**Approach:**
1. Print 4 ArUco markers on corners of a board
2. Detect ArUcos → get board's 3D pose relative to camera
3. Project a checkerboard pattern onto the board
4. Detect projected corners → map to 3D using known board pose
5. Use 3D points + projector pixels for stereo calibration

In [ ]:
import numpy as np
import cv2
from cv2 import aruco
import pickle
import os
from datetime import datetime
from capture_utils_v2 import CaptureSystem
import matplotlib.pyplot as plt

CALIB_DIR = r"C:\git\PhysicalAdverserialProj\procam_calibration_data"
calib_file = os.path.join(CALIB_DIR, 'procam_calibration.pkl')

# Ask user if camera calibration is needed
print("=" * 60)
print("CAMERA CALIBRATION OPTIONS")
print("=" * 60)
RUN_CAMERA_CALIBRATION = input("Run new camera calibration? (y/n, default=n): ").strip().lower() == 'y'

if RUN_CAMERA_CALIBRATION:
    print("\n→ Will run camera calibration using ArUco board")
    camera_matrix = None
    camera_dist = None
else:
    # Load existing camera calibration
    print("\n→ Loading existing camera calibration...")
    with open(calib_file, 'rb') as f:
        existing_calib = pickle.load(f)
    
    camera_matrix = existing_calib['camera_matrix']
    camera_dist = existing_calib['camera_dist']
    
    print("Loaded camera calibration:")
    print(f"  Camera Matrix:\n{camera_matrix}")
    print(f"  Distortion: {camera_dist.ravel()[:5]}...")

In [ ]:
# Initialize capture system
system = CaptureSystem()
PRJ_W, PRJ_H = system.screen_res
print(f"Projector resolution: {PRJ_W}x{PRJ_H}")

## Generate ChArUco Board for Printing (A4)

The ChArUco board will be placed on one side of the 70x50 cm board.
The projected checkerboard will be on the white area.

In [ ]:
# Generate ChArUco board for A4 printing
# This board will be placed on one side of a 70x50 cm board
# The projected pattern will be on the other (white) side

# ============================================================
# CHARUCO BOARD CONFIGURATION
# ============================================================

# A4 dimensions in mm (landscape orientation for more squares)
A4_WIDTH_MM = 297
A4_HEIGHT_MM = 210
MARGIN_MM = 15  # Margin from paper edge

# ChArUco board parameters
# FEWER, LARGER squares = more stable pose estimation (less jitter)
CHARUCO_SQUARES_X = 5  # Number of squares in X direction (was 9)
CHARUCO_SQUARES_Y = 4  # Number of squares in Y direction (was 6)

# Calculate square size to fit A4 with margins
usable_width = A4_WIDTH_MM - 2 * MARGIN_MM
usable_height = A4_HEIGHT_MM - 2 * MARGIN_MM
CHARUCO_SQUARE_SIZE_MM = min(usable_width / CHARUCO_SQUARES_X, usable_height / CHARUCO_SQUARES_Y)
CHARUCO_SQUARE_SIZE_MM = int(CHARUCO_SQUARE_SIZE_MM)  # Round down to integer

# ArUco marker is slightly smaller than the square (0.8x for good detection)
CHARUCO_MARKER_SIZE_MM = int(CHARUCO_SQUARE_SIZE_MM * 0.8)

# Use 4x4 dictionary (simple, robust)
CHARUCO_DICT = aruco.DICT_4X4_100
charuco_dict = aruco.getPredefinedDictionary(CHARUCO_DICT)

# Create ChArUco board - use MILLIMETERS to match rest of calibration code
charuco_board = aruco.CharucoBoard(
    (CHARUCO_SQUARES_X, CHARUCO_SQUARES_Y),
    float(CHARUCO_SQUARE_SIZE_MM),   # Keep in mm (not meters!)
    float(CHARUCO_MARKER_SIZE_MM),   # Keep in mm
    charuco_dict
)

# ============================================================
# GENERATE PRINTABLE IMAGE
# ============================================================
DPI = 300
MM_TO_INCH = 1 / 25.4

# Image size in pixels
img_width_px = int(A4_WIDTH_MM * MM_TO_INCH * DPI)
img_height_px = int(A4_HEIGHT_MM * MM_TO_INCH * DPI)

# Generate the board image
charuco_img = charuco_board.generateImage((img_width_px, img_height_px), marginSize=int(MARGIN_MM * MM_TO_INCH * DPI))

# Add text for reference
cv2.putText(charuco_img, f"ChArUco {CHARUCO_SQUARES_X}x{CHARUCO_SQUARES_Y} - Square: {CHARUCO_SQUARE_SIZE_MM}mm", 
            (50, img_height_px - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, 0, 2)

# Save for printing
charuco_output_path = os.path.join(CALIB_DIR, 'charuco_board_A4.png')
cv2.imwrite(charuco_output_path, charuco_img)

print("=" * 60)
print("CHARUCO BOARD FOR PROCAM CALIBRATION")
print("=" * 60)
print(f"Board dimensions: {CHARUCO_SQUARES_X} x {CHARUCO_SQUARES_Y} squares")
print(f"Square size: {CHARUCO_SQUARE_SIZE_MM} mm")
print(f"Marker size: {CHARUCO_MARKER_SIZE_MM} mm")
print(f"Number of markers: {(CHARUCO_SQUARES_X * CHARUCO_SQUARES_Y) // 2}")
print(f"Number of checkerboard corners: {(CHARUCO_SQUARES_X - 1) * (CHARUCO_SQUARES_Y - 1)}")
print(f"\nPaper size: A4 landscape ({A4_WIDTH_MM} x {A4_HEIGHT_MM} mm)")
print(f"Saved to: {charuco_output_path}")
print(f"\nPrint at {DPI} DPI for correct physical dimensions")
print("=" * 60)

# ============================================================
# PHYSICAL BOARD SETUP INFO
# ============================================================
PHYSICAL_BOARD_WIDTH_MM = 700   # 70 cm
PHYSICAL_BOARD_HEIGHT_MM = 500  # 50 cm

print(f"\n📋 SETUP INSTRUCTIONS:")
print(f"   1. Print the ChArUco board on A4 paper")
print(f"   2. Stick it on one SIDE of your {PHYSICAL_BOARD_WIDTH_MM//10}x{PHYSICAL_BOARD_HEIGHT_MM//10} cm board")
print(f"   3. Leave the rest as white projection area")
print(f"   4. The projected checkerboard will land on the white area")
print(f"\n   Layout (top view of 70x50 cm board):")
print(f"   ┌──────────────────────────────────────────┐")
print(f"   │  ChArUco    │                            │")
print(f"   │   (A4)      │     Projection Area        │")
print(f"   │             │                            │")
print(f"   └──────────────────────────────────────────┘")

# Display the board
plt.figure(figsize=(14, 10))
plt.imshow(charuco_img, cmap='gray')
plt.title(f'ChArUco Board ({CHARUCO_SQUARES_X}x{CHARUCO_SQUARES_Y}) - Print on A4 Landscape')
plt.axis('off')
plt.show()

## Step 1: Generate ArUco Board for Printing

Print this image and attach to a flat rigid board (cardboard, foam board, etc.)

In [ ]:
# ArUco board configuration
# L-shaped pattern: 4 markers along top edge + 2 more down the left edge = 6 total
# Leaves center free for projection

ARUCO_DICT = aruco.DICT_4X4_50  # Simple dictionary with 50 markers
aruco_dict = aruco.getPredefinedDictionary(ARUCO_DICT)

# A4 paper dimensions: 297mm x 210mm (landscape orientation)
PAPER_WIDTH_MM = 297  # Long side
PAPER_HEIGHT_MM = 210  # Short side

# Backwards compatibility aliases
BOARD_WIDTH_MM = PAPER_WIDTH_MM
BOARD_HEIGHT_MM = PAPER_HEIGHT_MM

# Physical dimensions (in mm)
MARKER_SIZE_MM = 45  # Slightly bigger than before (was 40)
MARGIN_MM = 13  # Margin from paper edge

# Marker IDs for L-shape (6 markers)
# Layout (landscape A4):
#   M0 --- M1 --- M2 --- M3  (top edge, 4 markers)
#   |
#   M4
#   |
#   M5
MARKER_IDS = [0, 1, 2, 3, 4, 5]

# Calculate spacing
# Top row: 4 markers evenly distributed along width
usable_width = PAPER_WIDTH_MM - 2 * MARGIN_MM - MARKER_SIZE_MM
top_spacing = usable_width / 3  # 3 gaps between 4 markers

# Left column: 3 markers evenly distributed along height (M0 is shared)
usable_height = PAPER_HEIGHT_MM - 2 * MARGIN_MM - MARKER_SIZE_MM
left_spacing = usable_height / 2  # 2 gaps between 3 markers (M0, M4, M5)

# Board coordinate system: origin at paper center
paper_half_w = PAPER_WIDTH_MM / 2
paper_half_h = PAPER_HEIGHT_MM / 2

# Marker center positions (from top-left corner of paper, then convert to centered coords)
def paper_to_board_coords(x_from_left, y_from_top):
    """Convert from paper coords (top-left origin) to board coords (center origin)."""
    x = x_from_left - paper_half_w
    y = y_from_top - paper_half_h
    return np.array([x, y, 0], dtype=np.float32)

# Top row markers (M0, M1, M2, M3)
top_y = MARGIN_MM + MARKER_SIZE_MM / 2
marker_centers = {
    0: paper_to_board_coords(MARGIN_MM + MARKER_SIZE_MM/2, top_y),
    1: paper_to_board_coords(MARGIN_MM + MARKER_SIZE_MM/2 + top_spacing, top_y),
    2: paper_to_board_coords(MARGIN_MM + MARKER_SIZE_MM/2 + 2*top_spacing, top_y),
    3: paper_to_board_coords(MARGIN_MM + MARKER_SIZE_MM/2 + 3*top_spacing, top_y),
}

# Left column markers (M4, M5) - M0 is already at top-left
left_x = MARGIN_MM + MARKER_SIZE_MM / 2
marker_centers[4] = paper_to_board_coords(left_x, MARGIN_MM + MARKER_SIZE_MM/2 + left_spacing)
marker_centers[5] = paper_to_board_coords(left_x, MARGIN_MM + MARKER_SIZE_MM/2 + 2*left_spacing)

# For compatibility with existing code
half_m = MARKER_SIZE_MM / 2

# Each marker's 4 corners (in marker's local frame, CCW from top-left)
marker_corner_offsets = np.array([
    [-half_m, -half_m, 0],  # Top-left
    [+half_m, -half_m, 0],  # Top-right
    [+half_m, +half_m, 0],  # Bottom-right
    [-half_m, +half_m, 0],  # Bottom-left
])

# Full 3D corners for all markers
all_marker_corners_3d = {}
for mid, center in marker_centers.items():
    corners = center + marker_corner_offsets
    all_marker_corners_3d[mid] = corners.astype(np.float32)

print(f"L-shaped ArUco Board Configuration:")
print(f"  Paper size: {PAPER_WIDTH_MM} x {PAPER_HEIGHT_MM} mm (A4 landscape)")
print(f"  Marker size: {MARKER_SIZE_MM} mm")
print(f"  Number of markers: {len(MARKER_IDS)}")
print(f"  Top row spacing: {top_spacing:.1f} mm")
print(f"  Left column spacing: {left_spacing:.1f} mm")
print(f"  Marker IDs: {MARKER_IDS}")

In [ ]:
# Generate printable L-shaped ArUco board image
DPI = 300
MM_TO_INCH = 1 / 25.4

# A4 paper size in pixels
total_w_px = int(PAPER_WIDTH_MM * MM_TO_INCH * DPI)
total_h_px = int(PAPER_HEIGHT_MM * MM_TO_INCH * DPI)
marker_size_px = int(MARKER_SIZE_MM * MM_TO_INCH * DPI)
margin_px = int(MARGIN_MM * MM_TO_INCH * DPI)

# Create white board image
board_img = np.ones((total_h_px, total_w_px), dtype=np.uint8) * 255

# Calculate pixel positions for markers
top_spacing_px = int(top_spacing * MM_TO_INCH * DPI)
left_spacing_px = int(left_spacing * MM_TO_INCH * DPI)

# Place top row markers (M0, M1, M2, M3)
top_y_px = margin_px
for i in range(4):
    x_px = margin_px + i * top_spacing_px
    marker = aruco.generateImageMarker(aruco_dict, MARKER_IDS[i], marker_size_px)
    board_img[top_y_px:top_y_px+marker_size_px, x_px:x_px+marker_size_px] = marker

# Place left column markers (M4, M5)
left_x_px = margin_px
for i, mid in enumerate([4, 5]):
    y_px = margin_px + (i + 1) * left_spacing_px
    marker = aruco.generateImageMarker(aruco_dict, MARKER_IDS[mid], marker_size_px)
    board_img[y_px:y_px+marker_size_px, left_x_px:left_x_px+marker_size_px] = marker

# Draw L-shape outline to show the projection area
outline_thickness = 2
# Top edge line
cv2.line(board_img, 
         (margin_px + marker_size_px, margin_px + marker_size_px // 2),
         (total_w_px - margin_px, margin_px + marker_size_px // 2),
         128, outline_thickness)
# Left edge line
cv2.line(board_img,
         (margin_px + marker_size_px // 2, margin_px + marker_size_px),
         (margin_px + marker_size_px // 2, total_h_px - margin_px),
         128, outline_thickness)

# Add text instructions
cv2.putText(board_img, f"L-shaped ArUco Board - {len(MARKER_IDS)} markers", 
            (margin_px + marker_size_px + 20, total_h_px - margin_px), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, 0, 2)
cv2.putText(board_img, f"Marker size: {MARKER_SIZE_MM}mm", 
            (margin_px + marker_size_px + 20, total_h_px - margin_px - 30), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, 0, 1)

# Save for printing
output_path = os.path.join(CALIB_DIR, 'aruco_board_L_shape.png')
cv2.imwrite(output_path, board_img)

print(f"L-shaped ArUco board saved to: {output_path}")
print(f"Print at {DPI} DPI for correct physical dimensions (A4 landscape)")
print(f"\nLayout:")
print(f"  M0 --- M1 --- M2 --- M3  (top edge)")
print(f"  |")
print(f"  M4")
print(f"  |")
print(f"  M5")
print(f"\nFree projection area: center and right side of paper")

plt.figure(figsize=(12, 8))
plt.imshow(board_img, cmap='gray')
plt.title('L-shaped ArUco Board for Printing (A4 Landscape)')
plt.axis('off')
plt.show()

## Step 1b: Camera Intrinsic Calibration (Optional)

If you selected to run camera calibration, capture multiple images of a PRINTED checkerboard from different angles.
This is the standard approach for camera intrinsic calibration.

In [ ]:
# Camera Intrinsic Calibration using Printed Checkerboard
# Skip this cell if RUN_CAMERA_CALIBRATION is False

if not RUN_CAMERA_CALIBRATION:
    print("Skipping camera calibration - using existing calibration")
else:
    import time
    
    # Checkerboard parameters (ADJUST THESE TO MATCH YOUR PRINTED PATTERN)
    CHECKERBOARD_ROWS = 6  # Number of inner corners per row
    CHECKERBOARD_COLS = 9  # Number of inner corners per column
    SQUARE_SIZE_MM = 25.0  # Size of each square in mm
    
    # Prepare object points (3D points in real world space)
    objp = np.zeros((CHECKERBOARD_ROWS * CHECKERBOARD_COLS, 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD_COLS, 0:CHECKERBOARD_ROWS].T.reshape(-1, 2)
    objp *= SQUARE_SIZE_MM
    
    camera_obj_points = []  # 3D points
    camera_img_points = []  # 2D points in image
    camera_images = []
    
    print("=" * 60)
    print("CAMERA INTRINSIC CALIBRATION")
    print("=" * 60)
    print(f"Checkerboard: {CHECKERBOARD_COLS}x{CHECKERBOARD_ROWS} inner corners")
    print(f"Square size: {SQUARE_SIZE_MM}mm")
    print("\nHold the PRINTED checkerboard in front of the camera.")
    print("Move it to different positions and angles.")
    print("Press 'c' to capture when corners are detected (shown in green).")
    print("Press 'q' when done (need 10-15 captures).")
    print("=" * 60)
    
    cv2.namedWindow("Camera Calibration", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Camera Calibration", 960, 720)
    
    capture_count = 0
    
    while True:
        ret, frame = system.cap.read()
        if not ret:
            continue
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Find checkerboard corners
        found, corners = cv2.findChessboardCorners(
            gray, (CHECKERBOARD_COLS, CHECKERBOARD_ROWS),
            cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE
        )
        
        display = frame.copy()
        
        if found:
            # Refine corner positions
            corners_refined = cv2.cornerSubPix(
                gray, corners, (11, 11), (-1, -1),
                (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
            )
            cv2.drawChessboardCorners(display, (CHECKERBOARD_COLS, CHECKERBOARD_ROWS), corners_refined, found)
            cv2.putText(display, "Checkerboard DETECTED - Press 'c' to capture", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        else:
            cv2.putText(display, "Searching for checkerboard...", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
        cv2.putText(display, f"Captures: {capture_count}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        
        cv2.imshow("Camera Calibration", display)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') and found:
            camera_obj_points.append(objp.copy())
            camera_img_points.append(corners_refined)
            camera_images.append(frame.copy())
            capture_count += 1
            print(f"✓ Captured image {capture_count}")
            time.sleep(0.3)
        elif key == ord('q'):
            break
    
    cv2.destroyWindow("Camera Calibration")
    print(f"\n✓ Captured {capture_count} images for camera calibration")
    
    # Compute camera intrinsics
    if len(camera_obj_points) >= 5:
        h, w = camera_images[0].shape[:2]
        
        ret_cam, camera_matrix, camera_dist, camera_rvecs, camera_tvecs = cv2.calibrateCamera(
            camera_obj_points, camera_img_points, (w, h), None, None
        )
        
        print(f"\n{'='*60}")
        print("CAMERA INTRINSIC CALIBRATION RESULTS")
        print(f"{'='*60}")
        print(f"RMS reprojection error: {ret_cam:.4f} pixels")
        print(f"\nCamera Matrix (K):")
        print(camera_matrix)
        print(f"\nDistortion Coefficients:")
        print(camera_dist.ravel())
        print(f"\nFocal lengths: fx={camera_matrix[0,0]:.2f}, fy={camera_matrix[1,1]:.2f}")
        print(f"Principal point: cx={camera_matrix[0,2]:.2f}, cy={camera_matrix[1,2]:.2f}")
        
        # Save camera calibration
        save_cam_calib = input("\nSave new camera calibration? (y/n): ").strip().lower() == 'y'
        if save_cam_calib:
            camera_calib_data = {
                'camera_matrix': camera_matrix,
                'camera_dist': camera_dist,
                'camera_rms': ret_cam,
                'image_size': (w, h),
                'checkerboard_size': (CHECKERBOARD_COLS, CHECKERBOARD_ROWS),
                'square_size_mm': SQUARE_SIZE_MM,
                'num_captures': len(camera_images),
                'timestamp': datetime.now().isoformat(),
            }
            with open(calib_file, 'wb') as f:
                pickle.dump(camera_calib_data, f)
            print(f"✓ Camera calibration saved to: {calib_file}")
    else:
        print("⚠ Need at least 5 captures for calibration!")

## Step 2: Generate Projected Checkerboard Pattern

This pattern will be projected onto the ArUco board

In [ ]:
# Generate projector checkerboard pattern (larger for better detection)
PRJ_CHECKER_COLS = 9   # Inner corners (was 7)
PRJ_CHECKER_ROWS = 6   # Inner corners (was 5)
SQUARE_SIZE_PIX = 100  # Projector pixels per square (was 80)

# Calculate pattern dimensions
pattern_w = (PRJ_CHECKER_COLS + 1) * SQUARE_SIZE_PIX
pattern_h = (PRJ_CHECKER_ROWS + 1) * SQUARE_SIZE_PIX

# Center the pattern
offset_x = (PRJ_W - pattern_w) // 2
offset_y = (PRJ_H - pattern_h) // 2

# Create checkerboard pattern
proj_checker = np.ones((PRJ_H, PRJ_W, 3), dtype=np.uint8) * 128  # Gray background

for row in range(PRJ_CHECKER_ROWS + 1):
    for col in range(PRJ_CHECKER_COLS + 1):
        x1 = offset_x + col * SQUARE_SIZE_PIX
        y1 = offset_y + row * SQUARE_SIZE_PIX
        x2 = x1 + SQUARE_SIZE_PIX
        y2 = y1 + SQUARE_SIZE_PIX
        
        color = 255 if (row + col) % 2 == 0 else 0
        proj_checker[y1:y2, x1:x2] = color

# Projector corner coordinates (what the projector "sees")
proj_corners = []
for row in range(PRJ_CHECKER_ROWS):
    for col in range(PRJ_CHECKER_COLS):
        px = offset_x + (col + 1) * SQUARE_SIZE_PIX
        py = offset_y + (row + 1) * SQUARE_SIZE_PIX
        proj_corners.append([px, py])

proj_corners = np.array(proj_corners, dtype=np.float32)

# Also create inverted and white patterns
proj_checker_inv = 255 - proj_checker.copy()
proj_checker_inv[proj_checker == 128] = 128
white_img = np.ones((PRJ_H, PRJ_W, 3), dtype=np.uint8) * 255

print(f"Projector checkerboard: {PRJ_CHECKER_COLS}x{PRJ_CHECKER_ROWS} inner corners")
print(f"Square size: {SQUARE_SIZE_PIX} pixels")
print(f"Pattern size: {pattern_w}x{pattern_h} pixels")

plt.figure(figsize=(12, 7))
plt.imshow(proj_checker)
plt.scatter(proj_corners[:, 0], proj_corners[:, 1], c='r', s=10, marker='+')
plt.title('Projected Checkerboard Pattern')
plt.show()

## Step 3: Detection Functions

In [ ]:
# ChArUco detector - uses BOTH ArUco markers AND checkerboard corners!
aruco_params = aruco.DetectorParameters()
# Speed optimizations for real-time detection
aruco_params.cornerRefinementMethod = aruco.CORNER_REFINE_NONE  # Faster, rely on ChArUco corners instead
aruco_params.adaptiveThreshWinSizeMin = 5
aruco_params.adaptiveThreshWinSizeMax = 21
aruco_params.adaptiveThreshWinSizeStep = 4

aruco_detector = aruco.ArucoDetector(charuco_dict, aruco_params)

# Store previous pose for temporal consistency
prev_rvec = None
prev_tvec = None

def detect_aruco_board(frame, camera_matrix, camera_dist):
    """
    Detect ChArUco board using BOTH ArUco markers AND checkerboard corners.
    The checkerboard corners provide sub-pixel accuracy for stable pose.
    Returns: success, rvec, tvec, detected_data
    """
    global prev_rvec, prev_tvec
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Step 1: Fast ArUco marker detection
    marker_corners, marker_ids, rejected = aruco_detector.detectMarkers(gray)
    
    if marker_ids is None or len(marker_ids) < 2:
        return False, None, None, None
    
    # Step 2: Interpolate ChArUco corners from detected markers
    # Use CharucoDetector for newer OpenCV (4.7+)
    charuco_corners, charuco_ids, _, _ = aruco.CharucoDetector(charuco_board).detectBoard(
        gray, markerCorners=marker_corners, markerIds=marker_ids
    )
    
    if charuco_ids is None or len(charuco_ids) < 4:  # Need at least 4 corners for stable pose
        return False, None, None, (marker_corners, marker_ids)
    
    # Use ChArUco corners for pose estimation - MUCH more accurate!
    # Each corner has a known 3D position on the board
    obj_points = charuco_board.getChessboardCorners()[charuco_ids.flatten()]
    img_points = charuco_corners.reshape(-1, 2)
    
    # Use ITERATIVE solver with previous pose as initial guess for stability
    if prev_rvec is not None and prev_tvec is not None:
        success, rvec, tvec = cv2.solvePnP(
            obj_points, img_points, camera_matrix, camera_dist,
            rvec=prev_rvec.copy(), tvec=prev_tvec.copy(),
            useExtrinsicGuess=True,
            flags=cv2.SOLVEPNP_ITERATIVE
        )
    else:
        success, rvec, tvec = cv2.solvePnP(
            obj_points, img_points, camera_matrix, camera_dist,
            flags=cv2.SOLVEPNP_ITERATIVE
        )
    
    if success:
        # Validate pose - check Z is positive (board in front of camera)
        if tvec[2, 0] > 0:
            prev_rvec = rvec.copy()
            prev_tvec = tvec.copy()
            # Return charuco corners for visualization
            return True, rvec, tvec, (marker_corners, marker_ids, charuco_corners, charuco_ids)
        else:
            prev_rvec = None
            prev_tvec = None
            return False, None, None, (marker_corners, marker_ids)
    else:
        prev_rvec = None
        prev_tvec = None
        return False, None, None, (marker_corners, marker_ids)


def detect_projected_checker(frame, prj_cols, prj_rows):
    """Detect projected checkerboard corners in camera image."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Apply CLAHE for better contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE + cv2.CALIB_CB_FAST_CHECK
    found, corners = cv2.findChessboardCorners(gray, (prj_cols, prj_rows), flags)
    
    if found:
        corners = cv2.cornerSubPix(
            gray, corners, (11, 11), (-1, -1),
            (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        )
        
        # Ensure consistent corner ordering - first corner should be top-left
        # Compare first and last corner positions
        first_corner = corners[0, 0]
        last_corner = corners[-1, 0]
        
        # If first corner is to the right of or below the last corner, flip the array
        # This ensures corners always go from top-left to bottom-right
        if first_corner[0] + first_corner[1] > last_corner[0] + last_corner[1]:
            corners = corners[::-1].copy()
    
    return found, corners, gray


def camera_pixel_to_board_3d(pixel, rvec, tvec, camera_matrix, camera_dist):
    """
    Convert camera pixel to 3D point on the board plane (Z=0 in board coords).
    This is the key function that connects camera and projector!
    """
    # Undistort the point
    pixel_undist = cv2.undistortPoints(
        np.array([[pixel]], dtype=np.float32),
        camera_matrix, camera_dist, P=camera_matrix
    )[0, 0]
    
    # Get rotation matrix
    R, _ = cv2.Rodrigues(rvec)
    
    # Camera ray in camera coordinates
    fx, fy = camera_matrix[0, 0], camera_matrix[1, 1]
    cx, cy = camera_matrix[0, 2], camera_matrix[1, 2]
    
    ray_cam = np.array([
        (pixel_undist[0] - cx) / fx,
        (pixel_undist[1] - cy) / fy,
        1.0
    ])
    
    # Transform ray to board coordinates
    # The board plane is Z=0 in board coords
    # Ray: P = t_cam + lambda * ray_cam (in camera coords)
    # Transform to board coords: P_board = R^T @ (P - tvec)
    # We want P_board[2] = 0 (on the board plane)
    
    # Camera position in board coordinates
    cam_pos_board = -R.T @ tvec.flatten()
    
    # Ray direction in board coordinates  
    ray_board = R.T @ ray_cam
    
    # Find intersection with Z=0 plane
    # cam_pos_board[2] + lambda * ray_board[2] = 0
    if abs(ray_board[2]) < 1e-6:
        return None  # Ray parallel to board
    
    lambda_val = -cam_pos_board[2] / ray_board[2]
    
    if lambda_val < 0:
        return None  # Intersection behind camera
    
    point_3d = cam_pos_board + lambda_val * ray_board
    point_3d[2] = 0  # Ensure exactly on plane
    
    return point_3d

print("Detection functions ready")

## Step 4: Capture Session

Hold the printed ArUco board in front of the projector.
The checkerboard pattern will be projected onto the board.
Capture at different poses.

In [ ]:
cv2.destroyAllWindows()

In [ ]:
# Interactive capture session
import time
import random

NUM_POSES = 50
all_captures = []

PROJECTOR_X_OFFSET = 1920  # Try 1920 for 1080p main monitor (was 1280)

# Randomize pattern position between captures for better calibration coverage
RANDOMIZE_PATTERN_POSITION = False  # Set to False to use fixed centered pattern
MAX_OFFSET_PIX = 150  # Maximum random offset in each direction (pixels)
PATTERN_BACKGROUND_LEVEL = 128  # Gray level for pattern background (0=black, 128=gray, 255=white)
ARROW_KEY_STEP = 10  # Pixels to move pattern per arrow key press

def generate_pattern_at_offset(offset_x_pix, offset_y_pix, invert=False):
    """Generate checkerboard pattern at a specific offset from center."""
    # Create checkerboard pattern
    pattern = np.ones((PRJ_H, PRJ_W, 3), dtype=np.uint8) * PATTERN_BACKGROUND_LEVEL  # Dim gray background
    
    curr_offset_x = (PRJ_W - pattern_w) // 2 + offset_x_pix
    curr_offset_y = (PRJ_H - pattern_h) // 2 + offset_y_pix
    
    for row in range(PRJ_CHECKER_ROWS + 1):
        for col in range(PRJ_CHECKER_COLS + 1):
            x1 = curr_offset_x + col * SQUARE_SIZE_PIX
            y1 = curr_offset_y + row * SQUARE_SIZE_PIX
            x2 = x1 + SQUARE_SIZE_PIX
            y2 = y1 + SQUARE_SIZE_PIX
            
            color = 255 if (row + col) % 2 == 0 else 0
            if invert:
                color = 255 - color
            pattern[max(0,y1):min(PRJ_H,y2), max(0,x1):min(PRJ_W,x2)] = color
    
    # Compute corner coordinates for this offset
    corners = []
    for row in range(PRJ_CHECKER_ROWS):
        for col in range(PRJ_CHECKER_COLS):
            px = curr_offset_x + (col + 1) * SQUARE_SIZE_PIX
            py = curr_offset_y + (row + 1) * SQUARE_SIZE_PIX
            corners.append([px, py])
    
    return pattern, np.array(corners, dtype=np.float32)

# Initialize with centered pattern
current_offset_x = 0
current_offset_y = 0
current_proj_checker, current_proj_corners = generate_pattern_at_offset(0, 0)
current_proj_checker_inv, _ = generate_pattern_at_offset(0, 0, invert=True)

print("=" * 60)
print("ARUCO + PROJECTED CHECKERBOARD CALIBRATION")
print("=" * 60)
print(f"Randomize pattern position: {RANDOMIZE_PATTERN_POSITION}")
if RANDOMIZE_PATTERN_POSITION:
    print(f"Max offset: ±{MAX_OFFSET_PIX} pixels")

# Setup fullscreen projector window
cv2.namedWindow("Projector", cv2.WND_PROP_FULLSCREEN)
cv2.moveWindow("Projector", PROJECTOR_X_OFFSET, 0)
cv2.setWindowProperty("Projector", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

# Show pattern
cv2.imshow("Projector", current_proj_checker)
cv2.waitKey(500)

print("\nHold the PRINTED ARUCO BOARD in front of the projector.")
print("The checkerboard pattern should land on the white area of the board.")
print(f"\nNeed {NUM_POSES} good poses with varied positions and angles.")
print("Press 'c' to capture when BOTH are detected (shown in green).")
print("Press 's' to skip this pose and get a new pattern position.")
print("Press 'i' to invert pattern if detection is difficult.")
print("Press 'r' to randomize pattern position manually.")
if not RANDOMIZE_PATTERN_POSITION:
    print("Use ARROW KEYS or W/A/X/D to move pattern position.")
print("Press 'q' when done.")
print("=" * 60)

# Camera preview
cv2.namedWindow("Calibration", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Calibration", 960, 720)

use_inverted = False
capture_idx = 0

while capture_idx < NUM_POSES:
    ret, frame = system.cap.read()
    if not ret:
        continue
    
    display = frame.copy()
    
    # Detect ArUco markers
    aruco_ok, rvec, tvec, aruco_data = detect_aruco_board(frame, camera_matrix, camera_dist)
    
    # Detect projected checkerboard
    checker_ok, checker_corners, gray = detect_projected_checker(
        frame, PRJ_CHECKER_COLS, PRJ_CHECKER_ROWS
    )
    
    # Draw ArUco markers and ChArUco corners
    if aruco_data is not None:
        if len(aruco_data) == 4:
            # Full ChArUco detection: (marker_corners, marker_ids, charuco_corners, charuco_ids)
            corners, ids, charuco_corners_det, charuco_ids_det = aruco_data
            aruco.drawDetectedMarkers(display, corners, ids)
            # Draw ChArUco corners (red dots for sub-pixel accurate corners)
            aruco.drawDetectedCornersCharuco(display, charuco_corners_det, charuco_ids_det, (0, 0, 255))
        else:
            # Fallback: only marker corners
            corners, ids = aruco_data
            aruco.drawDetectedMarkers(display, corners, ids)
        if aruco_ok:
            cv2.drawFrameAxes(display, camera_matrix, camera_dist, rvec, tvec, 50)
    
    # Draw checkerboard corners
    if checker_ok:
        cv2.drawChessboardCorners(display, (PRJ_CHECKER_COLS, PRJ_CHECKER_ROWS), 
                                   checker_corners, checker_ok)
    
    # Status - show ChArUco corner count for debugging
    if aruco_data is not None and len(aruco_data) == 4:
        n_corners = len(aruco_data[3])  # charuco_ids_det
        aruco_status = f"ChArUco: {n_corners} corners" if aruco_ok else f"ChArUco: {n_corners} (need 4+)"
    else:
        aruco_status = "ChArUco: OK" if aruco_ok else "ChArUco: NOT FOUND"
    checker_status = "Checker: OK" if checker_ok else "Checker: NOT FOUND"
    
    both_ok = aruco_ok and checker_ok
    color = (0, 255, 0) if both_ok else (0, 0, 255)
    
    cv2.putText(display, f"{aruco_status} | {checker_status} [{capture_idx}/{NUM_POSES}]",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    
    if both_ok:
        cv2.putText(display, "Press 'c' to CAPTURE", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # Show current pattern offset
    cv2.putText(display, f"Offset: ({current_offset_x}, {current_offset_y}) px", 
                (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    cv2.imshow("Calibration", display)
    key = cv2.waitKeyEx(30)  # Use waitKeyEx for arrow key support
    
    if key == ord('q'):
        break
    elif key == ord('i'):
        use_inverted = not use_inverted
        cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
        cv2.waitKey(100)
        time.sleep(0.3)
        print(f"Pattern: {'Inverted' if use_inverted else 'Normal'}")
    elif key == ord('r'):
        # Manually randomize pattern position
        current_offset_x = random.randint(-MAX_OFFSET_PIX, MAX_OFFSET_PIX)
        current_offset_y = random.randint(-MAX_OFFSET_PIX, MAX_OFFSET_PIX)
        current_proj_checker, current_proj_corners = generate_pattern_at_offset(current_offset_x, current_offset_y)
        current_proj_checker_inv, _ = generate_pattern_at_offset(current_offset_x, current_offset_y, invert=True)
        cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
        cv2.waitKey(100)
        print(f"Pattern offset: ({current_offset_x}, {current_offset_y}) pixels")
    elif key == ord('s'):
        # Skip this pose - just generate new pattern position without saving
        current_offset_x = random.randint(-MAX_OFFSET_PIX, MAX_OFFSET_PIX)
        current_offset_y = random.randint(-MAX_OFFSET_PIX, MAX_OFFSET_PIX)
        current_proj_checker, current_proj_corners = generate_pattern_at_offset(current_offset_x, current_offset_y)
        current_proj_checker_inv, _ = generate_pattern_at_offset(current_offset_x, current_offset_y, invert=True)
        cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
        cv2.waitKey(100)
        print(f"Skipped pose → New pattern offset: ({current_offset_x}, {current_offset_y}) pixels")
    # Arrow key controls for manual pattern positioning
    # Windows arrow key codes with waitKeyEx: Up=2490368, Down=2621440, Left=2424832, Right=2555904
    elif key == 2490368 or key == ord('w'):  # Up arrow or W
        current_offset_y -= ARROW_KEY_STEP
        current_proj_checker, current_proj_corners = generate_pattern_at_offset(current_offset_x, current_offset_y)
        current_proj_checker_inv, _ = generate_pattern_at_offset(current_offset_x, current_offset_y, invert=True)
        cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
        print(f"↑ Pattern offset: ({current_offset_x}, {current_offset_y}) pixels")
    elif key == 2621440 or key == ord('x'):  # Down arrow or X
        current_offset_y += ARROW_KEY_STEP
        current_proj_checker, current_proj_corners = generate_pattern_at_offset(current_offset_x, current_offset_y)
        current_proj_checker_inv, _ = generate_pattern_at_offset(current_offset_x, current_offset_y, invert=True)
        cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
        print(f"↓ Pattern offset: ({current_offset_x}, {current_offset_y}) pixels")
    elif key == 2424832 or key == ord('a'):  # Left arrow or A
        current_offset_x -= ARROW_KEY_STEP
        current_proj_checker, current_proj_corners = generate_pattern_at_offset(current_offset_x, current_offset_y)
        current_proj_checker_inv, _ = generate_pattern_at_offset(current_offset_x, current_offset_y, invert=True)
        cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
        print(f"← Pattern offset: ({current_offset_x}, {current_offset_y}) pixels")
    elif key == 2555904 or key == ord('d'):  # Right arrow or D
        current_offset_x += ARROW_KEY_STEP
        current_proj_checker, current_proj_corners = generate_pattern_at_offset(current_offset_x, current_offset_y)
        current_proj_checker_inv, _ = generate_pattern_at_offset(current_offset_x, current_offset_y, invert=True)
        cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
        print(f"→ Pattern offset: ({current_offset_x}, {current_offset_y}) pixels")
    elif key == ord('c') and both_ok:
        # Capture this pose
        all_captures.append({
            'frame': frame.copy(),
            'rvec': rvec.copy(),
            'tvec': tvec.copy(),
            'cam_corners': checker_corners.copy(),
            'proj_corners': current_proj_corners.copy(),  # Use current pattern corners
            'aruco_data': aruco_data,
            'pattern_offset': (current_offset_x, current_offset_y),  # Store offset for reference
        })
        capture_idx += 1
        print(f"✓ Captured pose {capture_idx} (pattern offset: {current_offset_x}, {current_offset_y})")
        
        # Randomize pattern position for next capture if enabled
        if RANDOMIZE_PATTERN_POSITION:
            current_offset_x = random.randint(-MAX_OFFSET_PIX, MAX_OFFSET_PIX)
            current_offset_y = random.randint(-MAX_OFFSET_PIX, MAX_OFFSET_PIX)
            current_proj_checker, current_proj_corners = generate_pattern_at_offset(current_offset_x, current_offset_y)
            current_proj_checker_inv, _ = generate_pattern_at_offset(current_offset_x, current_offset_y, invert=True)
            cv2.imshow("Projector", current_proj_checker_inv if use_inverted else current_proj_checker)
            cv2.waitKey(100)
            print(f"  → Next pattern offset: ({current_offset_x}, {current_offset_y}) pixels")
        
        time.sleep(0.5)

cv2.destroyWindow("Calibration")
cv2.imshow("Projector", white_img)
cv2.waitKey(100)

print(f"\n✓ Captured {len(all_captures)} poses")

In [ ]:
# Visualize captured poses
n_captures = len(all_captures)
cols = min(4, n_captures)
rows = (n_captures + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
axes = np.atleast_2d(axes).flatten()

for i, capture in enumerate(all_captures):
    ax = axes[i]
    frame_rgb = cv2.cvtColor(capture['frame'], cv2.COLOR_BGR2RGB)
    corners = capture['cam_corners'].reshape(-1, 2)
    
    ax.imshow(frame_rgb)
    ax.scatter(corners[:, 0], corners[:, 1], c='r', s=5)
    ax.set_title(f"Pose {i+1}")
    ax.axis('off')

for i in range(n_captures, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Step 5: Compute 3D Points on Board Plane

For each captured pose:
1. Use ArUco to get board pose (R, t)
2. Map camera pixels (projected corners) to 3D points on the board plane

In [ ]:
# Convert camera detections to 3D points on the board plane
all_obj_points = []  # 3D points on board (in board coordinates)
all_cam_points = []  # Camera image points
all_prj_points = []  # Projector image points

for i, capture in enumerate(all_captures):
    rvec = capture['rvec']
    tvec = capture['tvec']
    cam_corners = capture['cam_corners'].reshape(-1, 2)
    prj_corners = capture['proj_corners'].reshape(-1, 2)
    
    obj_pts = []
    cam_pts = []
    prj_pts = []
    
    for j, cam_pt in enumerate(cam_corners):
        # Convert camera pixel to 3D point on board plane
        pt_3d = camera_pixel_to_board_3d(cam_pt, rvec, tvec, camera_matrix, camera_dist)
        
        if pt_3d is not None:
            obj_pts.append(pt_3d)
            cam_pts.append(cam_pt)
            prj_pts.append(prj_corners[j])
    
    if len(obj_pts) >= 10:  # Need enough points
        all_obj_points.append(np.array(obj_pts, dtype=np.float32))
        all_cam_points.append(np.array(cam_pts, dtype=np.float32))
        all_prj_points.append(np.array(prj_pts, dtype=np.float32))
        print(f"Pose {i+1}: {len(obj_pts)} valid 3D points")
    else:
        print(f"Pose {i+1}: SKIPPED (only {len(obj_pts)} valid points)")

print(f"\n✓ {len(all_obj_points)} poses ready for calibration")

In [ ]:
# Visualize the 3D points from all poses
fig = plt.figure(figsize=(12, 5))

# 2D view (X-Y plane = board plane)
ax1 = fig.add_subplot(121)
for i, obj_pts in enumerate(all_obj_points):
    ax1.scatter(obj_pts[:, 0], obj_pts[:, 1], s=5, label=f'Pose {i+1}')
ax1.set_xlabel('X (mm)')
ax1.set_ylabel('Y (mm)')
ax1.set_title('3D Points on Board (X-Y plane)')
ax1.set_aspect('equal')
ax1.legend()

# Projector points
ax2 = fig.add_subplot(122)
for i, prj_pts in enumerate(all_prj_points):
    ax2.scatter(prj_pts[:, 0], prj_pts[:, 1], s=5, label=f'Pose {i+1}')
ax2.set_xlabel('Projector X (pixels)')
ax2.set_ylabel('Projector Y (pixels)')
ax2.set_title('Projector Image Points')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## Step 6: Stereo Calibration

Now we have proper 3D-2D correspondences for both camera and projector!

In [ ]:
# Calibrate projector intrinsics
# Using improved flags from procam_yotam.py:
# - Projectors have minimal lens distortion (fix K1, K2, K3 to zero)
# - Projectors have no tangential distortion
# - Square pixel aspect ratio

cam_img_size = (all_captures[0]['frame'].shape[1], all_captures[0]['frame'].shape[0])
prj_img_size = (PRJ_W, PRJ_H)

print(f"Camera image size: {cam_img_size}")
print(f"Projector image size: {prj_img_size}")

# Initial projector matrix guess with principal point hint
# Tabletop projectors often have principal point in lower half of image
PROJECTOR_ORIENTATION = "lower_half"  # "lower_half", "upper_half", or "center"

if PROJECTOR_ORIENTATION == "lower_half":
    cy_correction = PRJ_H / 4  # Move principal point down
elif PROJECTOR_ORIENTATION == "upper_half":
    cy_correction = -PRJ_H / 4  # Move principal point up
else:
    cy_correction = 0

initial_proj_matrix = np.array([
    [np.mean([PRJ_W, PRJ_H]), 0, PRJ_W / 2],
    [0, np.mean([PRJ_W, PRJ_H]), PRJ_H / 2 + cy_correction],
    [0, 0, 1]
], dtype=np.float64)

print(f"\nInitial projector matrix (orientation={PROJECTOR_ORIENTATION}):")
print(initial_proj_matrix)

# Projector calibration flags (from procam_yotam.py)
# Projectors typically have:
# - No radial distortion (DLP/LCD optics are well-corrected)
# - No tangential distortion
# - Square pixels (aspect ratio = 1)
PROJ_CALIB_FLAGS = (
    cv2.CALIB_USE_INTRINSIC_GUESS +
    cv2.CALIB_FIX_ASPECT_RATIO +      # Square pixels
    cv2.CALIB_ZERO_TANGENT_DIST +     # No tangential distortion
    cv2.CALIB_FIX_K1 +                # No radial distortion
    cv2.CALIB_FIX_K2 +
    cv2.CALIB_FIX_K3
)

# Calibrate projector
ret_prj, projector_matrix, projector_dist, rvecs_prj, tvecs_prj = cv2.calibrateCamera(
    all_obj_points,
    all_prj_points,
    prj_img_size,
    initial_proj_matrix.copy(),
    np.zeros((1, 5)),  # Initial distortion = 0
    flags=PROJ_CALIB_FLAGS
)

print(f"\nProjector calibration RMS: {ret_prj:.4f}")
print(f"\nProjector Matrix:")
print(projector_matrix)
print(f"\nProjector Distortion: {projector_dist.ravel()[:5]}")
print(f"  (Should be [0, 0, 0, 0, 0] due to CALIB_FIX_K* and ZERO_TANGENT_DIST flags)")
print(f"\nFocal length: fx=fy={projector_matrix[0,0]:.1f} (aspect ratio fixed to 1)")
print(f"Principal point: cx={projector_matrix[0,2]:.1f}, cy={projector_matrix[1,2]:.1f}")

In [ ]:
# Dynamically detect and remove outlier poses based on reprojection error
# Uses IQR (Interquartile Range) method to identify outliers

print("=" * 50)
print("DETECTING OUTLIER POSES")
print("=" * 50)

# Compute per-pose projector reprojection errors
pose_errors = []
for i, (obj_pts, prj_pts) in enumerate(zip(all_obj_points, all_prj_points)):
    _, rvec_prj, tvec_prj = cv2.solvePnP(obj_pts, prj_pts, projector_matrix, np.zeros(5))
    prj_reproj, _ = cv2.projectPoints(obj_pts, rvec_prj, tvec_prj, projector_matrix, np.zeros(5))
    error = np.sqrt(np.sum((prj_pts - prj_reproj.reshape(-1, 2)) ** 2, axis=1)).mean()
    pose_errors.append(error)
    print(f"Pose {i+1}: {error:.2f} px")

pose_errors = np.array(pose_errors)

# IQR-based outlier detection
q1 = np.percentile(pose_errors, 25)
q3 = np.percentile(pose_errors, 75)
iqr = q3 - q1
upper_threshold = q3 + 1.5 * iqr  # Standard IQR multiplier

# Also use absolute threshold - poses with error > 4px are likely bad
abs_threshold = 4.0
threshold = min(upper_threshold, abs_threshold)

print(f"\nOutlier detection:")
print(f"  Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"  IQR threshold: {upper_threshold:.2f} px")
print(f"  Absolute threshold: {abs_threshold:.2f} px")
print(f"  Using threshold: {threshold:.2f} px")

# Find outliers
outlier_indices = [i for i, err in enumerate(pose_errors) if err > threshold]
print(f"\nOutlier poses (error > {threshold:.2f} px): {[i+1 for i in outlier_indices]}")

if len(outlier_indices) == 0:
    print("\n✓ No outliers detected! Using all poses.")
    filtered_obj_points = all_obj_points
    filtered_cam_points = all_cam_points
    filtered_prj_points = all_prj_points
    filtered_captures = all_captures
    ret_prj_filtered = ret_prj
    proj_mtx_filtered = projector_matrix
    proj_dst_filtered = projector_dist
else:
    print(f"\nRemoving {len(outlier_indices)} outlier poses")
    
    # Filter out outliers
    filtered_obj_points = [p for i, p in enumerate(all_obj_points) if i not in outlier_indices]
    filtered_cam_points = [p for i, p in enumerate(all_cam_points) if i not in outlier_indices]
    filtered_prj_points = [p for i, p in enumerate(all_prj_points) if i not in outlier_indices]
    filtered_captures = [c for i, c in enumerate(all_captures) if i not in outlier_indices]
    
    print(f"Remaining poses: {len(filtered_obj_points)}")
    
    # Recalibrate projector with same improved flags
    ret_prj_filtered, proj_mtx_filtered, proj_dst_filtered, _, _ = cv2.calibrateCamera(
        filtered_obj_points,
        filtered_prj_points,
        prj_img_size,
        initial_proj_matrix.copy(),
        np.zeros((1, 5)),
        flags=PROJ_CALIB_FLAGS
    )
    print(f"\nFiltered Projector RMS: {ret_prj_filtered:.4f} (was {ret_prj:.4f})")

# Recalibrate stereo with filtered data
ret_stereo_filtered, cam_mtx_f, cam_dst_f, prj_mtx_f, prj_dst_f, R_f, T_f, E_f, F_f = cv2.stereoCalibrate(
    filtered_obj_points,
    filtered_cam_points,
    filtered_prj_points,
    camera_matrix,
    camera_dist,
    proj_mtx_filtered,
    np.zeros(5),
    cam_img_size,
    flags=cv2.CALIB_FIX_INTRINSIC,
    criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-6)
)

print(f"\n{'='*50}")
print("FILTERED STEREO CALIBRATION RESULTS")
print(f"{'='*50}")
print(f"Stereo RMS Error: {ret_stereo_filtered:.4f} pixels")
print(f"Poses used: {len(filtered_obj_points)} (removed {len(outlier_indices)} outliers)")
print(f"\nRotation Matrix R:")
print(R_f)
print(f"\nTranslation Vector T (mm):")
print(T_f.ravel())
print(f"\nBaseline distance: {np.linalg.norm(T_f):.1f} mm")

# Update the calibration variables to use filtered results
R = R_f
T = T_f
E = E_f
F = F_f
projector_matrix = proj_mtx_filtered
projector_dist = proj_dst_filtered
all_obj_points = filtered_obj_points
all_cam_points = filtered_cam_points
all_prj_points = filtered_prj_points
all_captures = filtered_captures
ret_stereo = ret_stereo_filtered

print(f"\n✓ Calibration variables updated with filtered results")

In [ ]:
# Analyze pose quality and visualize aspect ratios
# This runs AFTER outlier detection, so pose_errors and outlier_indices are defined

import matplotlib.pyplot as plt

print("=" * 70)
print("POSE QUALITY ANALYSIS (after outlier filtering)")
print("=" * 70)

# Compute 3D point aspect ratios for all original poses
pose_analysis = []
for i, capture in enumerate(all_captures if 'filtered_captures' not in dir() else filtered_captures):
    rvec = capture['rvec']
    tvec = capture['tvec']
    cam_corners = capture['cam_corners'].reshape(-1, 2)
    
    obj_pts_3d = []
    for cam_pt in cam_corners:
        pt_3d = camera_pixel_to_board_3d(cam_pt, rvec, tvec, camera_matrix, camera_dist)
        if pt_3d is not None:
            obj_pts_3d.append(pt_3d)
    obj_pts_3d = np.array(obj_pts_3d)
    
    analysis = {'pose_idx': i + 1}
    analysis['distance_mm'] = tvec[2, 0]
    
    if len(obj_pts_3d) > 0:
        x_range = obj_pts_3d[:, 0].max() - obj_pts_3d[:, 0].min()
        y_range = obj_pts_3d[:, 1].max() - obj_pts_3d[:, 1].min()
        analysis['x_range_mm'] = x_range
        analysis['y_range_mm'] = y_range
        analysis['aspect_ratio'] = x_range / max(y_range, 0.1)
    else:
        analysis['x_range_mm'] = 0
        analysis['y_range_mm'] = 0
        analysis['aspect_ratio'] = 0
    
    pose_analysis.append(analysis)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Aspect ratio distribution
ax1 = axes[0]
aspect_ratios = [p['aspect_ratio'] for p in pose_analysis if p['aspect_ratio'] > 0]
ax1.hist(aspect_ratios, bins=20, alpha=0.7, color='green', edgecolor='black')
ax1.axvline(1.4, color='blue', linestyle='--', linewidth=2, label=f'Expected (7/5=1.4)')
ax1.axvline(np.mean(aspect_ratios), color='orange', linestyle='-', linewidth=2, label=f'Mean ({np.mean(aspect_ratios):.2f})')
ax1.set_xlabel('Aspect Ratio (X range / Y range)')
ax1.set_ylabel('Count')
ax1.set_title('3D Point Aspect Ratio Distribution')
ax1.legend()

# Distance vs pose index
ax2 = axes[1]
distances = [p['distance_mm'] for p in pose_analysis]
ax2.bar(range(1, len(distances)+1), distances, alpha=0.7, color='steelblue')
ax2.set_xlabel('Pose Index')
ax2.set_ylabel('Distance from Camera (mm)')
ax2.set_title('Capture Distance Distribution')
ax2.axhline(np.mean(distances), color='red', linestyle='--', label=f'Mean: {np.mean(distances):.0f}mm')
ax2.legend()

plt.tight_layout()
plt.show()

# Summary
print(f"\nPoses used: {len(pose_analysis)}")
print(f"Aspect ratio: mean={np.mean(aspect_ratios):.2f}, expected=1.4 (7x5 grid)")
print(f"Distance: mean={np.mean(distances):.0f}mm, range=[{min(distances):.0f}, {max(distances):.0f}]mm")

In [ ]:
# Analyze pose variation - check if board was stationary
import numpy as np

print("=" * 60)
print("POSE VARIATION ANALYSIS")
print("=" * 60)

# Extract all poses
rvecs = np.array([c['rvec'].flatten() for c in all_captures])
tvecs = np.array([c['tvec'].flatten() for c in all_captures])

# Translation statistics (in mm)
tvec_mean = tvecs.mean(axis=0)
tvec_std = tvecs.std(axis=0)
tvec_range = tvecs.max(axis=0) - tvecs.min(axis=0)

print(f"\nTranslation (tvec) statistics:")
print(f"  Mean:  X={tvec_mean[0]:.1f}, Y={tvec_mean[1]:.1f}, Z={tvec_mean[2]:.1f} mm")
print(f"  Std:   X={tvec_std[0]:.1f}, Y={tvec_std[1]:.1f}, Z={tvec_std[2]:.1f} mm")
print(f"  Range: X={tvec_range[0]:.1f}, Y={tvec_range[1]:.1f}, Z={tvec_range[2]:.1f} mm")

# Convert rvecs to degrees for easier interpretation
rvec_degrees = np.array([np.linalg.norm(r) * 180 / np.pi for r in rvecs])
rvec_mean = rvec_degrees.mean()
rvec_std = rvec_degrees.std()
rvec_range = rvec_degrees.max() - rvec_degrees.min()

print(f"\nRotation (rvec) statistics:")
print(f"  Mean angle: {rvec_mean:.1f}°")
print(f"  Std:        {rvec_std:.1f}°")
print(f"  Range:      {rvec_range:.1f}°")

# Check if poses are essentially identical (board didn't move)
TRANSLATION_THRESHOLD = 50  # mm - if std < 50mm, board barely moved
ROTATION_THRESHOLD = 5      # degrees

translation_varied = tvec_std.max() > TRANSLATION_THRESHOLD
rotation_varied = rvec_std > ROTATION_THRESHOLD

print(f"\n{'='*60}")
if not translation_varied and not rotation_varied:
    print("⚠️  WARNING: Board was STATIONARY during calibration!")
    print("   This means the calibration may only be accurate at this specific pose.")
    print("   For robust calibration, vary the board position and angle.")
elif not translation_varied:
    print("⚠️  WARNING: Board position barely changed (only rotation varied)")
    print("   Consider moving the board to different distances/positions.")
elif not rotation_varied:
    print("⚠️  WARNING: Board angle barely changed (only position varied)")
    print("   Consider tilting the board at different angles.")
else:
    print("✓ Good pose variation detected!")
print("=" * 60)

# Visualize pose distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# XY position scatter
axes[0].scatter(tvecs[:, 0], tvecs[:, 1], c=range(len(tvecs)), cmap='viridis', alpha=0.7)
axes[0].set_xlabel('X (mm)')
axes[0].set_ylabel('Y (mm)')
axes[0].set_title('Board XY Position')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Distance over captures
axes[1].plot(tvecs[:, 2], 'b.-')
axes[1].set_xlabel('Capture #')
axes[1].set_ylabel('Z Distance (mm)')
axes[1].set_title('Board Distance from Camera')
axes[1].grid(True, alpha=0.3)

# Rotation magnitude over captures
axes[2].plot(rvec_degrees, 'r.-')
axes[2].set_xlabel('Capture #')
axes[2].set_ylabel('Rotation Angle (°)')
axes[2].set_title('Board Rotation')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Stereo calibration
projector_dist_zero = np.zeros(5)  # Assume no projector distortion

ret_stereo, cam_mtx, cam_dst, prj_mtx, prj_dst, R, T, E, F = cv2.stereoCalibrate(
    all_obj_points,
    all_cam_points,
    all_prj_points,
    camera_matrix,
    camera_dist,
    projector_matrix,
    projector_dist_zero,
    cam_img_size,
    flags=cv2.CALIB_FIX_INTRINSIC,
    criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-6)
)

print(f"\n{'='*50}")
print("STEREO CALIBRATION RESULTS")
print(f"{'='*50}")
print(f"Stereo RMS Error: {ret_stereo:.4f} pixels")
print(f"\nRotation Matrix R:")
print(R)
print(f"\nTranslation Vector T (mm):")
print(T.ravel())

# Compute baseline distance
baseline = np.linalg.norm(T)
print(f"\nBaseline distance: {baseline:.1f} mm")

In [ ]:
# Save calibration
calibration_data = {
    'camera_matrix': camera_matrix,
    'camera_dist': camera_dist,
    'projector_matrix': projector_matrix,
    'projector_dist': projector_dist_zero,
    'R': R,
    'T': T,
    'E': E,
    'F': F,
    'stereo_rms_error': ret_stereo,
    'projector_rms_error': ret_prj,
    'num_poses': len(all_captures),
    'timestamp': datetime.now().isoformat(),
    'method': 'aruco_projected_checkerboard',
    'board_config': {
        'marker_size_mm': MARKER_SIZE_MM,
        'board_width_mm': BOARD_WIDTH_MM,
        'board_height_mm': BOARD_HEIGHT_MM,
    }
}

output_file = os.path.join(CALIB_DIR, 'procam_calibration_aruco.pkl')
with open(output_file, 'wb') as f:
    pickle.dump(calibration_data, f)

print(f"\n✓ Calibration saved to: {output_file}")
print(f"\nSummary:")
print(f"  Projector RMS: {ret_prj:.4f} pixels")
print(f"  Stereo RMS: {ret_stereo:.4f} pixels")
print(f"  Baseline: {baseline:.1f} mm")

In [ ]:
# Verify calibration with reprojection
total_cam_error = 0
total_prj_error = 0
total_points = 0

for i, (obj_pts, cam_pts, prj_pts) in enumerate(zip(all_obj_points, all_cam_points, all_prj_points)):
    # Camera reprojection
    _, rvec_cam, tvec_cam = cv2.solvePnP(obj_pts, cam_pts, camera_matrix, camera_dist)
    cam_reproj, _ = cv2.projectPoints(obj_pts, rvec_cam, tvec_cam, camera_matrix, camera_dist)
    cam_error = np.sqrt(np.sum((cam_pts - cam_reproj.reshape(-1, 2)) ** 2, axis=1)).mean()
    
    # Projector reprojection
    _, rvec_prj, tvec_prj = cv2.solvePnP(obj_pts, prj_pts, projector_matrix, projector_dist_zero)
    prj_reproj, _ = cv2.projectPoints(obj_pts, rvec_prj, tvec_prj, projector_matrix, projector_dist_zero)
    prj_error = np.sqrt(np.sum((prj_pts - prj_reproj.reshape(-1, 2)) ** 2, axis=1)).mean()
    
    print(f"Pose {i+1}: Camera={cam_error:.3f}px, Projector={prj_error:.3f}px")
    
    total_cam_error += cam_error * len(cam_pts)
    total_prj_error += prj_error * len(prj_pts)
    total_points += len(cam_pts)

print(f"\nOverall Camera reprojection: {total_cam_error / total_points:.3f} px")
print(f"Overall Projector reprojection: {total_prj_error / total_points:.3f} px")

In [ ]:
# Diagnose the high stereo RMS error
print("=" * 60)
print("DIAGNOSING HIGH STEREO RMS ERROR")
print("=" * 60)

# Issue 1: The 3D points are computed FROM camera observations using ArUco pose
# This means camera reprojection is 0 by construction - not a real test!
print("\n1. Camera reprojection is 0 because 3D points are derived from camera pixels")
print("   This is a circular dependency - not measuring actual camera calibration quality")

# Issue 2: Check projector distortion - extreme values suggest overfitting
print(f"\n2. Projector distortion coefficients: {projector_dist.ravel()}")
print("   These extreme values suggest overfitting!")

# Issue 3: Check if 3D points have reasonable scale
all_pts = np.vstack(all_obj_points)
print(f"\n3. 3D point statistics (mm):")
print(f"   X range: [{all_pts[:, 0].min():.1f}, {all_pts[:, 0].max():.1f}]")
print(f"   Y range: [{all_pts[:, 1].min():.1f}, {all_pts[:, 1].max():.1f}]")
print(f"   Z range: [{all_pts[:, 2].min():.1f}, {all_pts[:, 2].max():.1f}]")
print(f"   Expected board area: {BOARD_WIDTH_MM} x {BOARD_HEIGHT_MM} mm")

# Issue 4: The stereo calibration is trying to find R,T between camera and projector
# but using CALIB_FIX_INTRINSIC means it can't adjust for intrinsic errors
print(f"\n4. Stereo calibration used CALIB_FIX_INTRINSIC")
print(f"   Camera intrinsics from existing calibration - may have errors")
print(f"   If intrinsics are wrong, R and T cannot compensate")

# Let's try stereo calibration allowing some intrinsic refinement
print("\n" + "=" * 60)
print("TRYING IMPROVED STEREO CALIBRATION")
print("=" * 60)

# Method 1: Allow projector intrinsics to be refined
ret_stereo2, cam_mtx2, cam_dst2, prj_mtx2, prj_dst2, R2, T2, E2, F2 = cv2.stereoCalibrate(
    all_obj_points,
    all_cam_points,
    all_prj_points,
    camera_matrix.copy(),
    camera_dist.copy(),
    projector_matrix.copy(),
    np.zeros(5),
    cam_img_size,
    flags=cv2.CALIB_FIX_INTRINSIC,  # Keep this but check epipolar error instead
    criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-6)
)

# Method 2: Use direct homography-based approach for procam
# Compute homography from projector to camera for each pose
print("\n5. Computing per-pose homographies (projector → camera):")
homography_errors = []
for i, (cam_pts, prj_pts) in enumerate(zip(all_cam_points, all_prj_points)):
    H, mask = cv2.findHomography(prj_pts, cam_pts, cv2.RANSAC, 5.0)
    if H is not None:
        # Compute reprojection error using homography
        prj_pts_h = np.hstack([prj_pts, np.ones((len(prj_pts), 1))])
        cam_pts_pred = (H @ prj_pts_h.T).T
        cam_pts_pred = cam_pts_pred[:, :2] / cam_pts_pred[:, 2:3]
        error = np.sqrt(np.sum((cam_pts - cam_pts_pred) ** 2, axis=1)).mean()
        homography_errors.append(error)
        print(f"   Pose {i+1}: Homography error = {error:.2f} px")

print(f"\n   Average homography error: {np.mean(homography_errors):.2f} px")
print("   (This measures direct projector→camera mapping quality)")

# The real issue: stereo RMS measures 3D triangulation consistency
# If the 3D points themselves have errors (from ArUco pose), stereo RMS will be high
print("\n" + "=" * 60)
print("ROOT CAUSE: The 3D object points are computed from ArUco pose estimation,")
print("which may have errors. The stereo RMS measures how well camera and projector")
print("rays intersect at these 3D points - but the points may be wrong!")
print("=" * 60)

## Calibration Test: Virtual Square on Board

Project a white square that appears "stuck" to the ArUco board center.
The square should remain stable and square-shaped to an observer as the board moves.

In [ ]:
cv2.destroyAllWindows()

In [ ]:
# Test calibration: Project a CHECKERBOARD that fits between the ArUco markers
# Compare actual projection location vs expected location in camera view

import time

# Board dimensions from config: ArUco markers are at corners
# BOARD_WIDTH_MM = 75, BOARD_HEIGHT_MM = 90, MARKER_SIZE_MM = 15
# The inner area (between marker inner edges) is where we'll project

# Checkerboard that fits inside the ArUco markers - SMALL to fit comfortably
CHECKER_ROWS = 3  # Number of rows of squares
CHECKER_COLS = 3  # Number of columns of squares
CHECKER_W_MM = 25   # Width of checkerboard region in mm
CHECKER_H_MM = 30   # Height of checkerboard region in mm

def create_checkerboard_3d_points(rows, cols, width_mm, height_mm, offset_x=0, offset_y=0):
    """Create 3D points for all checkerboard squares (on Z=0 board plane)."""
    sq_w = width_mm / cols
    sq_h = height_mm / rows
    
    points = []
    colors = []
    
    start_x = -width_mm / 2 + offset_x
    start_y = -height_mm / 2 + offset_y
    
    for r in range(rows):
        for c in range(cols):
            x0 = start_x + c * sq_w
            y0 = start_y + r * sq_h
            x1 = x0 + sq_w
            y1 = y0 + sq_h
            
            corners = np.array([
                [x0, y0, 0],
                [x1, y0, 0],
                [x1, y1, 0],
                [x0, y1, 0]
            ], dtype=np.float32)
            
            points.append(corners)
            colors.append((r + c) % 2)
    
    return points, colors


def project_board_to_projector_3d(points_3d_board, rvec_cam, tvec_cam, R_stereo, T_stereo, proj_matrix, proj_dist=None):
    """Project 3D board points to projector pixels using proper 3D transformation."""
    if proj_dist is None:
        proj_dist = np.zeros(5)
    
    R_cam, _ = cv2.Rodrigues(rvec_cam)
    R_board_to_prj = R_stereo @ R_cam
    t_board_to_prj = R_stereo @ tvec_cam.flatten() + T_stereo.flatten()
    rvec_prj, _ = cv2.Rodrigues(R_board_to_prj)
    
    prj_pts, _ = cv2.projectPoints(
        points_3d_board, rvec_prj, t_board_to_prj.reshape(3, 1), proj_matrix, proj_dist
    )
    return prj_pts.reshape(-1, 2).astype(np.int32)


def project_board_to_projector_3d_inv(points_3d_board, rvec_cam, tvec_cam, R_stereo, T_stereo, proj_matrix, proj_dist=None):
    """Same but with INVERSE stereo transform."""
    if proj_dist is None:
        proj_dist = np.zeros(5)
    
    R_inv = R_stereo.T
    T_inv = -R_stereo.T @ T_stereo.flatten()
    
    R_cam, _ = cv2.Rodrigues(rvec_cam)
    R_board_to_prj = R_inv @ R_cam
    t_board_to_prj = R_inv @ tvec_cam.flatten() + T_inv
    rvec_prj, _ = cv2.Rodrigues(R_board_to_prj)
    
    prj_pts, _ = cv2.projectPoints(
        points_3d_board, rvec_prj, t_board_to_prj.reshape(3, 1), proj_matrix, proj_dist
    )
    return prj_pts.reshape(-1, 2).astype(np.int32)


print("=" * 60)
print("CALIBRATION TEST: Checkerboard Projection")
print("=" * 60)
print(f"\nProjecting {CHECKER_COLS}x{CHECKER_ROWS} checkerboard ({CHECKER_W_MM}x{CHECKER_H_MM}mm)")
print(f"Board: {BOARD_WIDTH_MM}x{BOARD_HEIGHT_MM}mm, markers: {MARKER_SIZE_MM}mm")
print("\nControls:")
print("  ARROW KEYS or W/A/X/D = move pattern (5mm steps)")
print("  +/- = scale, i = inverse, r = reset, q = quit")
print("=" * 60)

# Setup projector window  
cv2.namedWindow("Projector", cv2.WND_PROP_FULLSCREEN)
cv2.moveWindow("Projector", PROJECTOR_X_OFFSET, 0)
cv2.setWindowProperty("Projector", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

# Camera preview window
cv2.namedWindow("Camera", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Camera", 1280, 960)

offset_x_mm = 0.0
offset_y_mm = 0.0
scale = 1.0
use_inverse = False
frame_count = 0

print(f"\nCalibration data:")
print(f"  R:\n{R}")
print(f"  T: {T.ravel()} mm")
print(f"  Projector: fx={projector_matrix[0,0]:.0f}, fy={projector_matrix[1,1]:.0f}, cx={projector_matrix[0,2]:.0f}, cy={projector_matrix[1,2]:.0f}")

while True:
    ret, frame = system.cap.read()
    if not ret:
        continue
    
    display = frame.copy()
    frame_count += 1
    
    # Detect ArUco board
    aruco_ok, rvec, tvec, aruco_data = detect_aruco_board(frame, camera_matrix, camera_dist)
    
    # Create projector image - DIM GRAY background (less harsh on eyes)
    BACKGROUND_LEVEL = 60  # 0=black, 255=white
    proj_img = np.ones((PRJ_H, PRJ_W, 3), dtype=np.uint8) * BACKGROUND_LEVEL
    
    # Draw a border and center marker for reference
    cv2.rectangle(proj_img, (10, 10), (PRJ_W-10, PRJ_H-10), (100, 100, 100), 2)
    cv2.drawMarker(proj_img, (PRJ_W//2, PRJ_H//2), (100, 100, 100), cv2.MARKER_CROSS, 50, 2)
    
    if aruco_ok:
        # Create checkerboard squares
        curr_w = CHECKER_W_MM * scale
        curr_h = CHECKER_H_MM * scale
        squares_3d, square_colors = create_checkerboard_3d_points(
            CHECKER_ROWS, CHECKER_COLS, curr_w, curr_h, offset_x_mm, offset_y_mm
        )
        
        # Project each square
        all_prj_squares = []
        all_cam_squares = []
        
        for sq_corners in squares_3d:
            if use_inverse:
                prj_sq = project_board_to_projector_3d_inv(sq_corners, rvec, tvec, R, T, projector_matrix)
            else:
                prj_sq = project_board_to_projector_3d(sq_corners, rvec, tvec, R, T, projector_matrix)
            
            all_prj_squares.append(prj_sq)
            
            cam_sq, _ = cv2.projectPoints(sq_corners, rvec, tvec, camera_matrix, camera_dist)
            all_cam_squares.append(cam_sq.reshape(-1, 2).astype(np.int32))
        
        # Print debug info every 30 frames
        if frame_count % 30 == 1:
            print(f"\n--- Frame {frame_count} ---")
            print(f"Board tvec: {tvec.ravel()}")
            print(f"First square projector coords: {all_prj_squares[0]}")
            print(f"First square camera coords: {all_cam_squares[0]}")
        
        # Draw checkerboard on projector (BLACK and WHITE squares on gray background)
        for prj_sq, color in zip(all_prj_squares, square_colors):
            # Clip to valid range before drawing
            prj_sq_clipped = np.clip(prj_sq, -10000, 10000)
            fill_color = (20, 20, 20) if color == 0 else (235, 235, 235)  # Dark gray or light gray
            cv2.fillPoly(proj_img, [prj_sq_clipped], fill_color)
        
        # Add red border around the checkerboard for visibility
        all_prj_pts = np.vstack(all_prj_squares)
        prj_min = np.clip(all_prj_pts.min(axis=0), 0, [PRJ_W, PRJ_H])
        prj_max = np.clip(all_prj_pts.max(axis=0), 0, [PRJ_W, PRJ_H])
        cv2.rectangle(proj_img, tuple(prj_min), tuple(prj_max), (0, 0, 255), 3)
        
        # Show coordinates on projector
        mode_str = "INV" if use_inverse else "NORM"
        cv2.putText(proj_img, f"{mode_str} | PRJ:{all_prj_squares[0][0]}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
        # Draw EXPECTED location on camera (green outlines)
        for cam_sq, color in zip(all_cam_squares, square_colors):
            cv2.polylines(display, [cam_sq], True, (0, 255, 0), 2)
        
        # Draw bounding box of expected area
        all_cam_pts = np.vstack(all_cam_squares)
        if np.all(np.isfinite(all_cam_pts)):
            cam_min = all_cam_pts.min(axis=0).astype(int)
            cam_max = all_cam_pts.max(axis=0).astype(int)
            cv2.rectangle(display, tuple(cam_min), tuple(cam_max), (255, 255, 0), 2)
            cv2.putText(display, "EXPECTED", (cam_min[0], cam_min[1] - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        
        # Draw ArUco markers (handle both old 2-tuple and new 4-tuple format)
        if len(aruco_data) == 4:
            corners, ids, charuco_corners_det, charuco_ids_det = aruco_data
            aruco.drawDetectedMarkers(display, corners, ids)
            aruco.drawDetectedCornersCharuco(display, charuco_corners_det, charuco_ids_det, (0, 0, 255))
        else:
            corners, ids = aruco_data
            aruco.drawDetectedMarkers(display, corners, ids)
        cv2.drawFrameAxes(display, camera_matrix, camera_dist, rvec, tvec, 30)
        
        # Status on camera view
        mode = "INV" if use_inverse else "NORM"
        cv2.putText(display, f"{mode} | Scale:{scale:.1f} | Offset:({offset_x_mm:.0f},{offset_y_mm:.0f})", 
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(display, "Green=EXPECTED | Look for BLACK squares on projector", 
                    (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    else:
        cv2.putText(display, "ArUco NOT DETECTED - show the board", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        # Show "NO ARUCO" message on projector
        cv2.putText(proj_img, "NO ARUCO DETECTED", (PRJ_W//2 - 300, PRJ_H//2), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 4)
    
    cv2.imshow("Projector", proj_img)
    cv2.imshow("Camera", display)
    
    key = cv2.waitKeyEx(30)  # Use waitKeyEx for arrow key support on Windows
    
    if key == ord('q'):
        break
    elif key == ord('i'):
        use_inverse = not use_inverse
        print(f"Using {'INVERSE' if use_inverse else 'NORMAL'} stereo transform")
    elif key == ord('+') or key == ord('='):
        scale = min(scale + 0.2, 3.0)
        print(f"Scale: {scale:.1f}")
    elif key == ord('-') or key == ord('_'):
        scale = max(scale - 0.2, 0.2)
        print(f"Scale: {scale:.1f}")
    # Arrow keys (Windows codes with waitKeyEx) and WASD alternatives
    elif key == 2490368 or key == ord('w'):  # Up arrow or W
        offset_y_mm -= 5
        print(f"↑ Offset: ({offset_x_mm:.0f}, {offset_y_mm:.0f}) mm")
    elif key == 2621440 or key == ord('x'):  # Down arrow or X
        offset_y_mm += 5
        print(f"↓ Offset: ({offset_x_mm:.0f}, {offset_y_mm:.0f}) mm")
    elif key == 2424832 or key == ord('a'):  # Left arrow or A
        offset_x_mm -= 5
        print(f"← Offset: ({offset_x_mm:.0f}, {offset_y_mm:.0f}) mm")
    elif key == 2555904 or key == ord('d'):  # Right arrow or D
        offset_x_mm += 5
        print(f"→ Offset: ({offset_x_mm:.0f}, {offset_y_mm:.0f}) mm")
    elif key == ord('r'):
        offset_x_mm = offset_y_mm = 0
        scale = 1.0
        print("Reset to defaults")

cv2.destroyAllWindows()

print(f"\n✓ Test complete")
print(f"Used {'INVERSE' if use_inverse else 'NORMAL'} transform")
print(f"Final offset: ({offset_x_mm:.0f}, {offset_y_mm:.0f}) mm, scale: {scale:.1f}")

In [ ]:
cv2.destroyAllWindows()